<a href="https://colab.research.google.com/github/HiyoriVn/CSE_Machine-Learning/blob/main/BaiTap_3_26_3_30_GD_Perceptron.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bài tập 3.26 - 3.30: Gradient Descent và Perceptron

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, make_blobs
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Perceptron as SkPerceptron
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay)

plt.rcParams['figure.figsize'] = (7, 4.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## Bài 3.26 – Gradient Descent cho f(x) = x² − 4x + 5

**1. Đạo hàm**

f'(x) = 2x − 4

Đạo hàm bằng 0 tại x = 2, và f''(x) = 2 > 0 nên đây là điểm cực tiểu, f(2) = 1.

**2. Công thức cập nhật**

x(t+1) = x(t) − η·f'(x(t)) = x(t) − 0.2·(2x(t) − 4)

Rút gọn: x(t+1) − 2 = (1 − 2η)(x(t) − 2) = 0.6·(x(t) − 2).

In [ ]:
def f(x):
    return x**2 - 4*x + 5

def df(x):
    return 2*x - 4

x0, eta, n_steps = 5.0, 0.2, 4

xs = [x0]
for _ in range(n_steps):
    xs.append(xs[-1] - eta * df(xs[-1]))
xs = np.array(xs)

bang = pd.DataFrame({
    'bước t': range(n_steps + 1),
    'x(t)': xs,
    "f'(x(t))": df(xs),
    'f(x(t))': f(xs),
    '|x(t) - 2|': np.abs(xs - 2),
})
bang.round(6)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))

# đường parabol và các điểm GD đi qua
g = np.linspace(1, 5.5, 200)
ax[0].plot(g, f(g), color='steelblue', label='f(x)')
ax[0].plot(xs, f(xs), 'o-', color='crimson', label='các bước GD')
for t, (x, y) in enumerate(zip(xs, f(xs))):
    ax[0].annotate(f'x{t}', (x, y), textcoords='offset points', xytext=(6, 6))
ax[0].scatter([2], [1], marker='*', s=200, color='gold', edgecolor='k', zorder=5, label='cực tiểu (2, 1)')
ax[0].set_title('Đường đi của Gradient Descent')
ax[0].set_xlabel('x'); ax[0].set_ylabel('f(x)'); ax[0].legend()

# f theo bước
ax[1].plot(range(n_steps + 1), f(xs), 'o-', color='darkgreen')
ax[1].axhline(1, color='gray', ls='--', label='giá trị nhỏ nhất = 1')
ax[1].set_title('Giá trị hàm số theo bước')
ax[1].set_xlabel('bước t'); ax[1].set_ylabel('f(x(t))'); ax[1].legend()

# khoảng cách tới nghiệm (thang log cho thấy giảm theo cấp số nhân)
ax[2].semilogy(range(n_steps + 1), np.abs(xs - 2), 'o-', color='purple')
ax[2].set_title('Khoảng cách |x(t) − 2| (thang log)')
ax[2].set_xlabel('bước t'); ax[2].set_ylabel('|x(t) − 2|')

plt.tight_layout(); plt.show()

**4. Nhận xét**

- Giá trị hàm giảm đều: 10 → 4.24 → 2.17 → 1.42 → 1.15, tiến dần về giá trị nhỏ nhất là 1.
- x tiến dần về 2 từ phía bên phải, không vượt qua nghiệm, vì hệ số co là 1 − 2η = 0.6 nằm trong (0, 1).
- Khoảng cách tới nghiệm nhân với 0.6 sau mỗi bước (3 → 1.8 → 1.08 → 0.648 → 0.389), nên trên thang log là một đường thẳng, tức hội tụ tuyến tính.
- Bước nhảy nhỏ dần vì gradient nhỏ dần khi tới gần cực tiểu.
- Muốn hội tụ cần 0 < η < 1 (khi |1 − 2η| < 1). Với η = 0.5 thì nhảy thẳng tới x = 2 ngay bước đầu, còn η > 1 thì bị phân kỳ. Ô dưới thử so sánh.

In [ ]:
def gd(eta, x0=5.0, steps=10):
    out = [x0]
    for _ in range(steps):
        out.append(out[-1] - eta * df(out[-1]))
    return np.array(out)

for e, c in zip([0.05, 0.2, 0.5, 0.9, 1.05], ['tab:blue', 'tab:green', 'tab:orange', 'tab:red', 'black']):
    plt.plot(f(gd(e)), 'o-', label=f'η = {e}', color=c)
plt.yscale('log')
plt.xlabel('bước'); plt.ylabel('f(x)  (thang log)')
plt.title('Ảnh hưởng của learning rate')
plt.legend(); plt.show()

## Bài 3.27 – Dự đoán nhãn bằng Perceptron

w = [1, 2, −10]ᵀ, x = [3, 4, 1]ᵀ (thành phần cuối của x là 1 do đã thêm bias, tương ứng w₀ = −10).

Quy tắc dự đoán: ŷ = sign(wᵀx).

In [ ]:
w = np.array([1, 2, -10])
x = np.array([3, 4, 1])

z = w @ x
y_hat = 1 if z >= 0 else -1
y_true = -1

print(f'w^T x = {w[0]}*{x[0]} + {w[1]}*{x[1]} + ({w[2]})*{x[2]} = {z}')
print('nhãn dự đoán:', y_hat)
print('nhãn thực tế:', y_true)
print('bị phân lớp sai' if y_hat != y_true else 'phân lớp đúng')
print('y * w^T x =', y_true * z, '(âm nghĩa là sai)')

In [ ]:
# w1*x1 + w2*x2 + b = 0  ->  x1 + 2*x2 - 10 = 0
x1 = np.linspace(-2, 12, 200)
x2 = (10 - x1) / 2

fig, ax = plt.subplots()
ax.plot(x1, x2, 'k-', label='ranh giới  x1 + 2·x2 − 10 = 0')
ax.fill_between(x1, x2, 10, color='tab:blue', alpha=0.12, label='vùng dự đoán +1')
ax.fill_between(x1, -3, x2, color='tab:red', alpha=0.12, label='vùng dự đoán −1')
ax.scatter([3], [4], s=120, color='tab:red', marker='s', zorder=5, label='điểm x = (3, 4), nhãn thật −1')
ax.annotate('wᵀx = 1 > 0\n=> dự đoán +1', (3, 4), textcoords='offset points', xytext=(15, -35))
ax.arrow(0, 0, w[0], w[1], head_width=0.2, color='green', length_includes_head=True)
ax.annotate('w', (w[0], w[1]), textcoords='offset points', xytext=(5, 5), color='green')
ax.set_xlim(-2, 12); ax.set_ylim(-3, 9); ax.set_aspect('equal')
ax.set_xlabel('x1'); ax.set_ylabel('x2')
ax.set_title('Bài 3.27: điểm (3, 4) nằm phía dương của ranh giới')
ax.legend(loc='upper right', fontsize=8); plt.show()

**Kết luận:** wᵀx = 3 + 8 − 10 = 1 > 0, nên nhãn dự đoán là +1. Nhãn thực tế là −1 nên điểm này **bị phân lớp sai**.

## Bài 3.28 – Một bước cập nhật Perceptron

w = [−2, 1, 0]ᵀ, x = [2, 3, 1]ᵀ, y = 1.

Nếu điểm bị sai thì cập nhật: w ← w + η·y·x (lấy η = 1).

In [ ]:
w_old = np.array([-2, 1, 0])
x = np.array([2, 3, 1])
y = 1

z_old = w_old @ x
print('w^T x trước cập nhật =', z_old)
print('dự đoán =', 1 if z_old >= 0 else -1, '| nhãn thật =', y)
sai = y * z_old <= 0
print('bị phân lớp sai?', sai)

if sai:
    w_new = w_old + y * x
    z_new = w_new @ x
    print('w mới =', w_new)
    print('w^T x sau cập nhật =', z_new)
    print('dự đoán mới =', 1 if z_new >= 0 else -1)

In [ ]:
# vẽ ranh giới trước và sau cập nhật
def duong_thang(w, xs):
    # w0*x1 + w1*x2 + w2 = 0
    return -(w[0] * xs + w[2]) / w[1]

xs_ = np.linspace(-1, 5, 100)
fig, ax = plt.subplots()
ax.plot(xs_, duong_thang(w_old, xs_), 'r--', label='ranh giới cũ  w = [−2, 1, 0]')
ax.plot(xs_, duong_thang(w_new, xs_), 'g-', label='ranh giới mới  w = [0, 4, 1]')
ax.scatter([2], [3], s=120, color='tab:blue', zorder=5, label='điểm (2, 3), nhãn +1')
ax.annotate('trước: wᵀx = −1\nsau: wᵀx = 13', (2, 3), textcoords='offset points', xytext=(10, -35))
ax.set_xlim(-1, 5); ax.set_ylim(-2, 7)
ax.set_xlabel('x1'); ax.set_ylabel('x2')
ax.set_title('Bài 3.28: ranh giới dịch chuyển để đưa điểm về đúng phía')
ax.legend(fontsize=8); plt.show()

**Kết luận**

1. wᵀx = −4 + 3 + 0 = −1 < 0, dự đoán −1 nhưng y = +1, nên mẫu bị phân lớp sai.
2. Cập nhật: w_new = w + y·x = [−2, 1, 0] + [2, 3, 1] = [0, 4, 1].
3. w_newᵀx = 0·2 + 4·3 + 1·1 = 13 > 0, điểm đã được phân lớp đúng.

Có thể kiểm tra: giá trị wᵀx tăng đúng bằng y·‖x‖² = 4 + 9 + 1 = 14 (từ −1 lên 13).

## Bài 3.29 – Cài đặt lớp Perceptron

Lớp nhận nhãn bất kỳ gồm hai giá trị (ví dụ 0/1), bên trong đổi sang −1/+1 để cập nhật. Có tham số `pocket`: giữ lại bộ trọng số có ít lỗi nhất trong quá trình huấn luyện, hữu ích khi dữ liệu không phân tách tuyến tính hoàn toàn (Perceptron thường sẽ không dừng lại vì luôn có điểm sai).

In [ ]:
class Perceptron:
    def __init__(self, lr=1.0, n_epochs=100, shuffle=True, pocket=False, random_state=0):
        self.lr = lr
        self.n_epochs = n_epochs
        self.shuffle = shuffle
        self.pocket = pocket
        self.random_state = random_state

    def _them_bias(self, X):
        return np.hstack([X, np.ones((X.shape[0], 1))])

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)

        self.classes_ = np.unique(y)
        if len(self.classes_) != 2:
            raise ValueError('Chỉ hỗ trợ bài toán hai lớp')
        t = np.where(y == self.classes_[1], 1, -1)

        Xb = self._them_bias(X)
        rng = np.random.default_rng(self.random_state)
        w = np.zeros(Xb.shape[1])

        self.errors_ = []          # số điểm sai của w hiện tại ở mỗi epoch
        best_w, best_err = w.copy(), np.inf

        for _ in range(self.n_epochs):
            idx = rng.permutation(len(Xb)) if self.shuffle else np.arange(len(Xb))
            for i in idx:
                if t[i] * (Xb[i] @ w) <= 0:
                    w = w + self.lr * t[i] * Xb[i]

            err = int(np.sum(np.where(Xb @ w >= 0, 1, -1) != t))
            self.errors_.append(err)
            if err < best_err:
                best_err, best_w = err, w.copy()
            if err == 0:
                break

        self.w_ = best_w if self.pocket else w
        return self

    def decision_function(self, X):
        return self._them_bias(np.asarray(X, dtype=float)) @ self.w_

    def predict(self, X):
        s = self.decision_function(X)
        return np.where(s >= 0, self.classes_[1], self.classes_[0])

    def score(self, X, y):
        return np.mean(self.predict(X) == np.asarray(y))

In [ ]:
# thử trên dữ liệu 2 chiều tách được
X_demo, y_demo = make_blobs(n_samples=200, centers=[(-2, -2), (2, 2)], cluster_std=1.0, random_state=1)

model = Perceptron(n_epochs=50, random_state=0).fit(X_demo, y_demo)
print('w (gồm bias ở cuối):', model.w_.round(3))
print('độ chính xác:', model.score(X_demo, y_demo))
print('dự báo 3 điểm mới:', model.predict([[-3, -1], [0.5, 0.2], [3, 4]]))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

# ranh giới quyết định
gx, gy = np.meshgrid(np.linspace(-6, 6, 300), np.linspace(-6, 6, 300))
zz = model.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
ax[0].contourf(gx, gy, zz, alpha=0.2, cmap='coolwarm')
ax[0].scatter(X_demo[:, 0], X_demo[:, 1], c=y_demo, cmap='coolwarm', edgecolor='k', s=30)
ax[0].set_title('Ranh giới quyết định của Perceptron tự cài')
ax[0].set_xlabel('x1'); ax[0].set_ylabel('x2')

# số lỗi qua các epoch
ax[1].plot(range(1, len(model.errors_) + 1), model.errors_, 'o-')
ax[1].set_title('Số điểm phân lớp sai sau mỗi epoch')
ax[1].set_xlabel('epoch'); ax[1].set_ylabel('số điểm sai')
plt.tight_layout(); plt.show()

## Bài 3.30 – Phân lớp nhị phân bằng Perceptron

**Dữ liệu:** Breast Cancer Wisconsin (có sẵn trong scikit-learn, 569 mẫu, 30 đặc trưng). Bài toán là dự đoán khối u ác tính hay lành tính.

Lớp dương (1) là **ác tính**, vì bỏ sót ca ác tính (Recall thấp) nghiêm trọng hơn báo nhầm.

In [ ]:
data = load_breast_cancer()
X = data.data
y = 1 - data.target          # đổi để 1 = ác tính, 0 = lành tính

print('kích thước:', X.shape)
print('ác tính (1):', y.sum(), '| lành tính (0):', (y == 0).sum())

pd.Series(y).map({0: 'lành tính', 1: 'ác tính'}).value_counts().plot(
    kind='bar', color=['tab:green', 'tab:red'], rot=0)
plt.title('Phân bố nhãn'); plt.ylabel('số mẫu'); plt.show()

### Chia dữ liệu và chuẩn hóa

Chia 60% train / 20% validation / 20% test, có phân tầng theo nhãn. Validation dùng để chọn siêu tham số, test chỉ dùng một lần ở cuối. Scaler chỉ fit trên tập train để không rò rỉ thông tin.

In [ ]:
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=42)

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

print(len(X_train), len(X_val), len(X_test))

### Chọn siêu tham số

Vì khởi tạo w = 0 nên learning rate chỉ làm w co giãn theo tỉ lệ, không đổi dự đoán. Do đó mình cố định lr = 1 và chỉ thử số epoch, có/không dùng pocket và vài seed cho thứ tự xáo trộn. Tiêu chí chọn là F1 trên tập validation.

In [ ]:
def danh_gia(y_true, y_pred):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred),
        'F1-score': f1_score(y_true, y_pred),
    }

rows = []
for pocket in [False, True]:
    for n_ep in [5, 10, 20, 50, 100, 300]:
        for seed in range(5):
            m = Perceptron(n_epochs=n_ep, pocket=pocket, random_state=seed).fit(X_train_s, y_train)
            r = danh_gia(y_val, m.predict(X_val_s))
            rows.append({'pocket': pocket, 'n_epochs': n_ep, 'seed': seed, **r})

ket_qua = pd.DataFrame(rows)
tong_hop = ket_qua.groupby(['pocket', 'n_epochs'])[['Accuracy', 'Precision', 'Recall', 'F1-score']].mean().round(4)
tong_hop

In [ ]:
# F1 trung bình trên validation theo số epoch
fig, ax = plt.subplots()
for pocket, c in [(False, 'tab:orange'), (True, 'tab:blue')]:
    sub = tong_hop.loc[pocket]
    ax.plot(sub.index, sub['F1-score'], 'o-', color=c, label=f'pocket = {pocket}')
ax.set_xscale('log')
ax.set_xlabel('số epoch (thang log)'); ax.set_ylabel('F1 trung bình trên validation')
ax.set_title('Chọn số epoch và chế độ pocket')
ax.legend(); plt.show()

best = ket_qua.sort_values('F1-score', ascending=False).iloc[0]
print('cấu hình tốt nhất trên validation:')
print(best.to_string())

### Huấn luyện lại và đánh giá trên tập test

Sau khi chọn được cấu hình, gộp train và validation lại để huấn luyện (scaler cũng fit lại trên phần gộp này), rồi đánh giá trên test.

In [ ]:
X_full = np.vstack([X_train, X_val])
y_full = np.concatenate([y_train, y_val])
scaler_f = StandardScaler().fit(X_full)
Xf, Xt = scaler_f.transform(X_full), scaler_f.transform(X_test)

final = Perceptron(n_epochs=int(best['n_epochs']), pocket=bool(best['pocket']),
                   random_state=int(best['seed'])).fit(Xf, y_full)
y_pred = final.predict(Xt)

# so sánh với Perceptron của sklearn
sk = SkPerceptron(max_iter=1000, tol=None, random_state=0).fit(Xf, y_full)
y_pred_sk = sk.predict(Xt)

so_sanh = pd.DataFrame({
    'Perceptron tự cài': danh_gia(y_test, y_pred),
    'Perceptron sklearn': danh_gia(y_test, y_pred_sk),
}).T
so_sanh.round(4)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))

# 1. ma trận nhầm lẫn
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=['lành tính', 'ác tính']).plot(ax=ax[0], cmap='Blues', colorbar=False)
ax[0].set_title('Ma trận nhầm lẫn (test)'); ax[0].grid(False)

# 2. so sánh các độ đo
so_sanh.T.plot(kind='bar', ax=ax[1], rot=0, color=['tab:blue', 'tab:gray'])
ax[1].set_ylim(0.85, 1.01)
ax[1].set_title('Các độ đo trên tập test'); ax[1].legend(fontsize=8)

# 3. số lỗi train theo epoch
ax[2].plot(range(1, len(final.errors_) + 1), final.errors_, color='tab:red')
ax[2].set_title('Số mẫu train sai sau mỗi epoch')
ax[2].set_xlabel('epoch'); ax[2].set_ylabel('số mẫu sai')

plt.tight_layout(); plt.show()

In [ ]:
# chiếu xuống 2 chiều bằng PCA để nhìn ranh giới (chỉ mang tính minh họa)
pca = PCA(n_components=2).fit(Xf)
Z_train = pca.transform(Xf)
Z_test = pca.transform(Xt)

p2 = Perceptron(n_epochs=int(best['n_epochs']), pocket=True, random_state=0).fit(Z_train, y_full)

gx, gy = np.meshgrid(np.linspace(Z_train[:, 0].min() - 1, Z_train[:, 0].max() + 1, 300),
                     np.linspace(Z_train[:, 1].min() - 1, Z_train[:, 1].max() + 1, 300))
zz = p2.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.contourf(gx, gy, zz, alpha=0.2, cmap='coolwarm')
sc = ax.scatter(Z_test[:, 0], Z_test[:, 1], c=y_test, cmap='coolwarm', edgecolor='k', s=35)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title(f'Tập test trên 2 thành phần chính (PCA giữ {pca.explained_variance_ratio_.sum():.0%} phương sai)')
ax.legend(*sc.legend_elements(), title='nhãn (1 = ác tính)', loc='upper right'); plt.show()
print('accuracy khi chỉ dùng 2 chiều PCA:', round(p2.score(Z_test, y_test), 4))

### Nhận xét

- Sau khi chuẩn hóa, Perceptron cho kết quả khá cao trên cả 4 độ đo, vì dữ liệu này gần như tách tuyến tính.
- Ma trận nhầm lẫn cho biết số ca ác tính bị bỏ sót (FN) và số ca lành tính bị báo nhầm (FP). Với bài toán y tế, FN đáng lo hơn nên Recall là độ đo cần chú ý nhất.
- Chế độ pocket và số epoch ảnh hưởng đến độ ổn định: Perceptron thuần không hội tụ khi dữ liệu không tách hoàn toàn, số lỗi dao động giữa các epoch (xem đồ thị số lỗi), còn pocket giữ lại nghiệm tốt nhất đã gặp.
- Nếu không chuẩn hóa, các đặc trưng có thang đo lớn (như area) sẽ lấn át các đặc trưng còn lại và mô hình học kém hơn nhiều.
- Do test chỉ có 114 mẫu nên chênh lệch một hai mẫu cũng làm các độ đo thay đổi vài phần trăm. Kết quả nên được xem là ước lượng chứ không phải con số cố định.
- Hạn chế của Perceptron là chỉ tạo ranh giới tuyến tính và không cho xác suất. Muốn tốt hơn có thể thử Logistic Regression hoặc SVM.